In [1]:
import os
os.chdir("/Users/varunsardana/PSTAT-134-Final-Project")
print(os.getcwd())

/Users/varunsardana/PSTAT-134-Final-Project


In [2]:
import sys
!{sys.executable} -m pip install nltk


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import re
import nltk

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("punkt")
nltk.download("punkt_tab")

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/varunsardana/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/varunsardana/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/varunsardana/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/varunsardana/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [4]:
import os
print(os.getcwd())

/Users/varunsardana/PSTAT-134-Final-Project


In [5]:
df = pd.read_csv("data/raw/reddit_raw.csv")

In [6]:

print(f"Loaded {len(df)} posts")
df.head(3)

Loaded 800 posts


,id,title,selftext,upvote_ratio,num_comments,is_self,edited,created_utc,subreddit
0,1nimpmu,DOJ Deletes Study Showing Domestic Terrorists ...,NaN,0.88,2681,False,False,1.758041e+09,technology
1,1nlju03,Disney+ cancellation page crashes as customers...,NaN,0.89,3300,False,False,1.758327e+09,technology
2,1nkl8b0,"Yes, Jimmy Kimmel’s suspension was government ...",NaN,0.81,3102,False,False,1.758232e+09,technology


In [7]:
# Fill empty selftext with empty string
df["selftext"] = df["selftext"].fillna("")

# Combine title and body into one column
df["full_text"] = df["title"] + " " + df["selftext"]

print("Sample full_text:")
print(df["full_text"][0])

# Here we are combining the title and the body,as some posts have no body text so we fill those with empty string first, then combine title + body into one single full_text column. This is the main text we'll use for all NLP analysis.

Sample full_text:
DOJ Deletes Study Showing Domestic Terrorists Are Most Often Right Wing 


In [8]:
def clean_text(text):
    text = text.lower()                        # make everything lowercase
    text = re.sub(r"http\S+", "", text)        # remove URLs
    text = re.sub(r"[^a-z\s]", "", text)       # remove numbers and special chars
    text = re.sub(r"\s+", " ", text).strip()   # remove extra spaces
    return text

df["full_text"] = df["full_text"].apply(clean_text)
print("Sample cleaned text:")
print(df["full_text"][0])

# cleaning the data overall

Sample cleaned text:
doj deletes study showing domestic terrorists are most often right wing


In [9]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]  # remove stopwords
    tokens = [lemmatizer.lemmatize(t) for t in tokens]   # lemmatize each word
    return " ".join(tokens)

df["full_text"] = df["full_text"].apply(preprocess)
print("Sample after stopwords + lemmatization:")
print(df["full_text"][0])

# removing the stop words and lemmatization

Sample after stopwords + lemmatization:
doj deletes study showing domestic terrorist often right wing


In [10]:
before = len(df)
df = df[df["full_text"].apply(lambda x: len(x.split()) >= 3)]
after = len(df)

print(f"Rows before: {before}")
print(f"Rows after dropping short posts: {after}")
print(f"Rows dropped: {before - after}")
print(f"\nPosts per subreddit:")
print(df["subreddit"].value_counts())

Rows before: 800
Rows after dropping short posts: 755
Rows dropped: 45

Posts per subreddit:
subreddit
technology         200
MachineLearning    200
artificial         188
singularity        167
Name: count, dtype: int64


In [11]:
df.to_csv("data/cleaned/reddit_cleaned.csv", index=False)
print(f"Saved {len(df)} cleaned posts to data/cleaned/reddit_cleaned.csv")

Saved 755 cleaned posts to data/cleaned/reddit_cleaned.csv
